In [1]:
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import numpy as np
import torch

In [2]:
data_root = Path("/network-volume/nhphuc/dataset")

datasets = {
    "15": [
        "stereomis/p2_1_1_247_s4",
        "stereomis/p2_6_5000_200_s4",
        "stereomis/p3_9100_368_s4",
        "stereomis/p3_11000_401_s4",
    ],
    "30": [
        "stereomis/p2_1_1_247_30fps",
        "stereomis/p2_6_5000_200_30fps",
        "stereomis/p3_9100_368",
        "stereomis/p3_11000_401",
    ],
    "60": [
        "stereomis/p2_1_1_247",
        "stereomis/p2_6_5000_200",
        "stereomis/p3_9100_368_60fps",
        "stereomis/p3_11000_401_60fps",
    ]
}

# Check if all datasets exist
for fps, dataset_list in datasets.items():
    for dataset in dataset_list:
        dataset_path = data_root / dataset
        if not dataset_path.exists():
            print(f"Dataset {dataset} does not exist at {dataset_path}")

In [12]:
import lpips
loss_fn = lpips.LPIPS(net='alex').cuda()  # best forward scores

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/root/miniconda3/envs/EndoFlow3/lib/python3.8/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/root/miniconda3/envs/EndoFlow3/lib/python3.8/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /root/miniconda3/envs/EndoFlow3/lib/python3.8/site-packages/lpips/weights/v0.1/alex.pth


/root/miniconda3/envs/EndoFlow3/lib/python3.8/site-packages/lpips/lpips.py:107: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.load_state_dict(torch.load(model_path, map

In [10]:
def PSNR(img1, img2):
    from utils.image_utils import psnr
    return psnr(img1, img2)

def SSIM(img1, img2):
    from utils.loss_utils import ssim
    return ssim(img1, img2)

def LPIPS(img1, img2):
    return loss_fn(img1, img2).item()

In [ ]:
def read_image(image_path):
    image = np.array(Image.open(image_path))
    image = (torch.from_numpy(image).float() / 255.0).permute(2, 0, 1)
    image = image.unsqueeze(0) # (1, 3, H, W)
    return image

In [16]:
from collections import defaultdict
psnrs = defaultdict(dict)
ssims = defaultdict(dict)
lpipss = defaultdict(dict)

for (fps, dataset_list) in datasets.items():
    print(f"FPS: {fps}")
    for dataset in dataset_list:
        dataset_path = data_root / dataset
        print(f"  Dataset: {dataset_path}")
        images = sorted(list((dataset_path / "images").glob("*.png")))
        print(f"    Number of images: {len(images)}")
        psnrs_list = []
        ssims_list = []
        lpipss_list = []
        for i in tqdm(range(len(images)-1)):
            img1 = read_image(images[i]).cuda().contiguous()
            img2 = read_image(images[i+1]).cuda().contiguous()
            # print(f"{img1.shape = }")
            psnrs_list.append(float(PSNR(img1, img2).item()))
            ssims_list.append(float(SSIM(img1, img2).item()))
            lpipss_list.append(LPIPS(img1, img2))

        psnrs[fps][dataset] = np.mean(psnrs_list)
        ssims[fps][dataset] = np.mean(ssims_list)
        lpipss[fps][dataset] = np.mean(lpipss_list)

print("PSNR:")
for fps, dataset_dict in psnrs.items():
    print(f"FPS: {fps}")
    for dataset, psnr in dataset_dict.items():
        print(f"  Dataset: {dataset}, PSNR: {psnr}")
print("SSIM:")
for fps, dataset_dict in ssims.items():
    print(f"FPS: {fps}")
    for dataset, ssim in dataset_dict.items():
        print(f"  Dataset: {dataset}, SSIM: {ssim}")
print("LPIPS:")
for fps, dataset_dict in lpipss.items():
    print(f"FPS: {fps}")
    for dataset, lpips in dataset_dict.items():
        print(f"  Dataset: {dataset}, LPIPS: {lpips}")

FPS: 15
  Dataset: /network-volume/nhphuc/dataset/stereomis/p2_1_1_247_s4
    Number of images: 62


100%|██████████| 61/61 [00:22<00:00,  2.75it/s]


  Dataset: /network-volume/nhphuc/dataset/stereomis/p2_6_5000_200_s4
    Number of images: 50


100%|██████████| 49/49 [00:16<00:00,  3.06it/s]


  Dataset: /network-volume/nhphuc/dataset/stereomis/p3_9100_368_s4
    Number of images: 92


...

  Dataset: /network-volume/nhphuc/dataset/stereomis/p3_9100_368_60fps
    Number of images: 368


100%|██████████| 367/367 [02:06<00:00,  2.91it/s]


  Dataset: /network-volume/nhphuc/dataset/stereomis/p3_11000_401_60fps
    Number of images: 401


100%|██████████| 400/400 [02:17<00:00,  2.91it/s]

PSNR:
FPS: 15
  Dataset: stereomis/p2_1_1_247_s4, PSNR: 27.349456849645396
  Dataset: stereomis/p2_6_5000_200_s4, PSNR: 30.868144015876613
  Dataset: stereomis/p3_9100_368_s4, PSNR: 27.55333248599545
  Dataset: stereomis/p3_11000_401_s4, PSNR: 23.01051467895508
FPS: 30
  Dataset: stereomis/p2_1_1_247_30fps, PSNR: 30.53331418541389
  Dataset: stereomis/p2_6_5000_200_30fps, PSNR: 34.36202527055837
  Dataset: stereomis/p3_9100_368, PSNR: 30.948829640456236
  Dataset: stereomis/p3_11000_401, PSNR: 25.269900369644166
FPS: 60
  Dataset: stereomis/p2_1_1_247, PSNR: 34.14070270507316
  Dataset: stereomis/p2_6_5000_200, PSNR: 38.27213270101116
  Dataset: stereomis/p3_9100_368_60fps, PSNR: 34.844637629121785
  Dataset: stereomis/p3_11000_401_60fps, PSNR: 28.33483284473419
SSIM:
FPS: 15
  Dataset: stereomis/p2_1_1_247_s4, SSIM: 0.675167623601976
  Dataset: stereomis/p2_6_5000_200_s4, SSIM: 0.8410108125939662
  Dataset: stereomis/p3_9100_368_s4, SSIM: 0.7585344511074025
  Dataset: stereomis/p3_110

In [ ]:
# PSNR:
# FPS: 15
#   Dataset: stereomis/p2_1_1_247_s4, PSNR: 27.349456849645396
#   Dataset: stereomis/p2_6_5000_200_s4, PSNR: 30.868144015876613
#   Dataset: stereomis/p3_9100_368_s4, PSNR: 27.55333248599545
#   Dataset: stereomis/p3_11000_401_s4, PSNR: 23.01051467895508
# FPS: 30
#   Dataset: stereomis/p2_1_1_247_30fps, PSNR: 30.53331418541389
#   Dataset: stereomis/p2_6_5000_200_30fps, PSNR: 34.36202527055837
#   Dataset: stereomis/p3_9100_368, PSNR: 30.948829640456236
#   Dataset: stereomis/p3_11000_401, PSNR: 25.269900369644166
# FPS: 60
#   Dataset: stereomis/p2_1_1_247, PSNR: 34.14070270507316
#   Dataset: stereomis/p2_6_5000_200, PSNR: 38.27213270101116
#   Dataset: stereomis/p3_9100_368_60fps, PSNR: 34.844637629121785
#   Dataset: stereomis/p3_11000_401_60fps, PSNR: 28.33483284473419
# SSIM:
# FPS: 15
#   Dataset: stereomis/p2_1_1_247_s4, SSIM: 0.675167623601976
#   Dataset: stereomis/p2_6_5000_200_s4, SSIM: 0.8410108125939662
#   Dataset: stereomis/p3_9100_368_s4, SSIM: 0.7585344511074025
#   Dataset: stereomis/p3_11000_401_s4, SSIM: 0.6678504440188407
# FPS: 30
#   Dataset: stereomis/p2_1_1_247_30fps, SSIM: 0.815867053784006
#   Dataset: stereomis/p2_6_5000_200_30fps, SSIM: 0.9165702323720912
#   Dataset: stereomis/p3_9100_368, SSIM: 0.8660227115688428
#   Dataset: stereomis/p3_11000_401, SSIM: 0.7818842449784279
# FPS: 60
#   Dataset: stereomis/p2_1_1_247, SSIM: 0.9096922414089607
#   Dataset: stereomis/p2_6_5000_200, SSIM: 0.9590488774093551
#   Dataset: stereomis/p3_9100_368_60fps, SSIM: 0.9346525032448834
#   Dataset: stereomis/p3_11000_401_60fps, SSIM: 0.8731408342719078
# LPIPS:
# FPS: 15
#   Dataset: stereomis/p2_1_1_247_s4, LPIPS: 0.049338438563415264
#   Dataset: stereomis/p2_6_5000_200_s4, LPIPS: 0.05060166508263471
#   Dataset: stereomis/p3_9100_368_s4, LPIPS: 0.04739546734880615
#   Dataset: stereomis/p3_11000_401_s4, LPIPS: 0.08451822567731142
# FPS: 30
#   Dataset: stereomis/p2_1_1_247_30fps, LPIPS: 0.03399079565594836
#   Dataset: stereomis/p2_6_5000_200_30fps, LPIPS: 0.031407960605892266
#   Dataset: stereomis/p3_9100_368, LPIPS: 0.029597478282581913
#   Dataset: stereomis/p3_11000_401, LPIPS: 0.051280401721596716
# FPS: 60
#   Dataset: stereomis/p2_1_1_247, LPIPS: 0.02063927255813184
#   Dataset: stereomis/p2_6_5000_200, LPIPS: 0.017362218752653154
#   Dataset: stereomis/p3_9100_368_60fps, LPIPS: 0.016987220683624862
#   Dataset: stereomis/p3_11000_401_60fps, LPIPS: 0.02958507838193327

In [ ]:
# psnrs = defaultdict(dict,
#             {'15': {'stereomis/p2_1_1_247_s4': 27.349456849645396,
#               'stereomis/p2_6_5000_200_s4': 30.868144015876613,
#               'stereomis/p3_9100_368_s4': 27.55333248599545,
#               'stereomis/p3_11000_401_s4': 23.01051467895508},
#              '30': {'stereomis/p2_1_1_247_30fps': 30.53331418541389,
#               'stereomis/p2_6_5000_200_30fps': 34.36202527055837,
#               'stereomis/p3_9100_368': 30.948829640456236,
#               'stereomis/p3_11000_401': 25.269900369644166},
#              '60': {'stereomis/p2_1_1_247': 34.14070270507316,
#               'stereomis/p2_6_5000_200': 38.27213270101116,
#               'stereomis/p3_9100_368_60fps': 34.844637629121785,
#               'stereomis/p3_11000_401_60fps': 28.33483284473419}})

defaultdict(dict,
            {'15': {'stereomis/p2_1_1_247_s4': 27.349456849645396,
              'stereomis/p2_6_5000_200_s4': 30.868144015876613,
              'stereomis/p3_9100_368_s4': 27.55333248599545,
              'stereomis/p3_11000_401_s4': 23.01051467895508},
             '30': {'stereomis/p2_1_1_247_30fps': 30.53331418541389,
              'stereomis/p2_6_5000_200_30fps': 34.36202527055837,
              'stereomis/p3_9100_368': 30.948829640456236,
              'stereomis/p3_11000_401': 25.269900369644166},
             '60': {'stereomis/p2_1_1_247': 34.14070270507316,
              'stereomis/p2_6_5000_200': 38.27213270101116,
              'stereomis/p3_9100_368_60fps': 34.844637629121785,
              'stereomis/p3_11000_401_60fps': 28.33483284473419}})

In [ ]:
# ssims = defaultdict(dict,
#             {'15': {'stereomis/p2_1_1_247_s4': 0.675167623601976,
#               'stereomis/p2_6_5000_200_s4': 0.8410108125939662,
#               'stereomis/p3_9100_368_s4': 0.7585344511074025,
#               'stereomis/p3_11000_401_s4': 0.6678504440188407},
#              '30': {'stereomis/p2_1_1_247_30fps': 0.815867053784006,
#               'stereomis/p2_6_5000_200_30fps': 0.9165702323720912,
#               'stereomis/p3_9100_368': 0.8660227115688428,
#               'stereomis/p3_11000_401': 0.7818842449784279},
#              '60': {'stereomis/p2_1_1_247': 0.9096922414089607,
#               'stereomis/p2_6_5000_200': 0.9590488774093551,
#               'stereomis/p3_9100_368_60fps': 0.9346525032448834,
#               'stereomis/p3_11000_401_60fps': 0.8731408342719078}})

defaultdict(dict,
            {'15': {'stereomis/p2_1_1_247_s4': 0.675167623601976,
              'stereomis/p2_6_5000_200_s4': 0.8410108125939662,
              'stereomis/p3_9100_368_s4': 0.7585344511074025,
              'stereomis/p3_11000_401_s4': 0.6678504440188407},
             '30': {'stereomis/p2_1_1_247_30fps': 0.815867053784006,
              'stereomis/p2_6_5000_200_30fps': 0.9165702323720912,
              'stereomis/p3_9100_368': 0.8660227115688428,
              'stereomis/p3_11000_401': 0.7818842449784279},
             '60': {'stereomis/p2_1_1_247': 0.9096922414089607,
              'stereomis/p2_6_5000_200': 0.9590488774093551,
              'stereomis/p3_9100_368_60fps': 0.9346525032448834,
              'stereomis/p3_11000_401_60fps': 0.8731408342719078}})

In [ ]:
# lpipss = defaultdict(dict,
#             {'15': {'stereomis/p2_1_1_247_s4': 0.049338438563415264,
#               'stereomis/p2_6_5000_200_s4': 0.05060166508263471,
#               'stereomis/p3_9100_368_s4': 0.04739546734880615,
#               'stereomis/p3_11000_401_s4': 0.08451822567731142},
#              '30': {'stereomis/p2_1_1_247_30fps': 0.03399079565594836,
#               'stereomis/p2_6_5000_200_30fps': 0.031407960605892266,
#               'stereomis/p3_9100_368': 0.029597478282581913,
#               'stereomis/p3_11000_401': 0.051280401721596716},
#              '60': {'stereomis/p2_1_1_247': 0.02063927255813184,
#               'stereomis/p2_6_5000_200': 0.017362218752653154,
#               'stereomis/p3_9100_368_60fps': 0.016987220683624862,
#               'stereomis/p3_11000_401_60fps': 0.02958507838193327}})

defaultdict(dict,
            {'15': {'stereomis/p2_1_1_247_s4': 0.049338438563415264,
              'stereomis/p2_6_5000_200_s4': 0.05060166508263471,
              'stereomis/p3_9100_368_s4': 0.04739546734880615,
              'stereomis/p3_11000_401_s4': 0.08451822567731142},
             '30': {'stereomis/p2_1_1_247_30fps': 0.03399079565594836,
              'stereomis/p2_6_5000_200_30fps': 0.031407960605892266,
              'stereomis/p3_9100_368': 0.029597478282581913,
              'stereomis/p3_11000_401': 0.051280401721596716},
             '60': {'stereomis/p2_1_1_247': 0.02063927255813184,
              'stereomis/p2_6_5000_200': 0.017362218752653154,
              'stereomis/p3_9100_368_60fps': 0.016987220683624862,
              'stereomis/p3_11000_401_60fps': 0.02958507838193327}})